In [1]:
# Cell 1: Install Dependencies & Mount Drive
import os

print("⚙️ Installing standard CPU inference stack...")
# Clean up dependencies—avoiding any heavy GPU backends entirely
!pip install --no-deps trl peft accelerator transformers datasets safetensors

print("📁 Mounting Google Drive to access your checkpoints...")
from google.colab import drive
drive.mount('/content/drive')

print("✅ CPU Environment successfully initialized.")

In [2]:
# Clear the incompatible torchao version from the environment
!pip uninstall -y torchao

In [3]:
# Cell 2: Define Checkpoint & Base Model Paths
import os

# Point exactly to your healthy checkpoint folder in Google Drive
CHECKPOINT_PATH = "/content/drive/MyDrive/SFT/modelS/mistral_dsa_buddy/checkpoints/checkpoint-100"

# Target the standard unquantized v0.3 base model instead of the 4bit variant
BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

if os.path.exists(CHECKPOINT_PATH):
    print(f"✅ Found healthy checkpoint directory at: {CHECKPOINT_PATH}")
else:
    print(f"❌ WARNING: Path not found! Double check your Google Drive directory structure.")

In [4]:
# Cell 3: Optimized Base Model and Layer the Checkpoint Adapter Natively on CPU
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Optimize CPU Threads (Colab CPU tiers usually give you 2 distinct compute cores)
torch.set_num_threads(2)
torch.set_num_interop_threads(2)

print("📦 Loading tokenizer directly from your checkpoint...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)

print("📦 Loading unquantized Mistral weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# 2. Activate native PyTorch CPU speed optimizations
print("⚡ Injecting BetterTransformer optimizations for CPU...")
try:
    base_model = base_model.to_bettertransformer()
except Exception as e:
    print(f"   (BetterTransformer notice: {e} - proceeding with native optimizations)")

print("🚀 Injecting your custom DSA Brain (checkpoint-100 LoRA adapter)...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)

model.eval()
print("🎯 Model is officially locked, loaded, and optimized for CPU inference!")

In [5]:
# Cell 4: Interactive DSA Interview Buddy Tester
from transformers import TextStreamer

# 1. Enter your DSA Question here
dsa_problem = "Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target."

# 2. Format the payload to strictly align with your custom structural training template
input_prompt = (
    f"<s>[INST] You are an expert DSA Interview Buddy. Analyze the problem, map your constraints "
    f"inside <thought>...</thought> tags, and then provide a friendly explanation with clean Python code.\n\n"
    f"Problem: {dsa_problem}\n\nPlease provide your reasoning and the solution strictly in Python. [/INST]\n"
)

# 3. Tokenize input strings
inputs = tokenizer([input_prompt], return_tensors="pt")

# 4. Initialize live stream printing
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

In [ ]:
print(f"\n🤖 DSA Buddy (HF CPU Optimized) is processing...\n" + "="*50)

# 5. Generate output block with strict inference optimizations
with torch.inference_mode():  # Faster than torch.no_grad() by completely skipping tensor overhead
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=512,   # Kept at 512 for quicker turnarounds on CPU threads
        use_cache=True,        # Reuses calculated key-value pairs to avoid heavy recalculations
        temperature=0.2,       # Keeps explanations focused and deterministic
        do_sample=True
    )